In [ ]:
# ===============================
# ✅ IMPORT LIBRARIES
# ===============================
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import time
import os

tf.get_logger().setLevel('ERROR')
# ===============================
# 📥 LOAD & PREPROCESS IMAGES
# ===============================
def load_img(path_to_img, max_dim=512):
    img = Image.open(path_to_img)
    img = img.convert('RGB')
    long = max(img.size)
    scale = max_dim / long
    img = img.resize((round(img.size[0]*scale), round(img.size[1]*scale)), Image.ANTIALIAS)
    img = tf.keras.preprocessing.image.img_to_array(img)
    img = tf.expand_dims(img, axis=0)
    return tf.keras.applications.vgg19.preprocess_input(img)

def tensor_to_image(tensor):
    tensor = tensor * 255
    tensor = tf.clip_by_value(tensor, 0, 255)
    tensor = np.array(tensor, dtype=np.uint8)
    if tensor.shape[0] > 1:
        tensor = tensor[0]
    return Image.fromarray(tensor)

def imshow(img, title=None):
    if len(img.shape) > 3:
        img = tf.squeeze(img, axis=0)
    plt.imshow(img / 255.0)
    if title:
        plt.title(title)
    plt.axis('off')


In [ ]:
# ===============================
# 🖼 LOAD CONTENT AND STYLE IMAGES
# ===============================
content_path = "/kaggle/input/style-transfer/content/content1.jpg"
style_path   = "/kaggle/input/style-transfer/style/style1.jpg"

content_image = load_img(content_path)
style_image = load_img(style_path)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
imshow(content_image, "Content Image")
plt.subplot(1, 2, 2)
imshow(style_image, "Style Image")
plt.show()
# ===============================
# 🧠 VGG MODEL SETUP
# ===============================
vgg = tf.keras.applications.VGG19(include_top=False, weights='imagenet')
vgg.trainable = False

content_layers = ['block5_conv2']
style_layers = [
    'block1_conv1', 'block2_conv1',
    'block3_conv1', 'block4_conv1',
    'block5_conv1'
]

num_content_layers = len(content_layers)
num_style_layers = len(style_layers)

def vgg_layers(layer_names):
    outputs = [vgg.get_layer(name).output for name in layer_names]
    model = tf.keras.Model([vgg.input], outputs)
    return model

def gram_matrix(input_tensor):
    result = tf.linalg.einsum('bijc,bijd->bcd', input_tensor, input_tensor)
    input_shape = tf.shape(input_tensor)
    num_locations = tf.cast(input_shape[1]*input_shape[2], tf.float32)
    return result / num_locations


In [ ]:
# ===============================
# 🔍 STYLE-CONTENT EXTRACTOR
# ===============================
class StyleContentModel(tf.keras.models.Model):
    def __init__(self, style_layers, content_layers):
        super().__init__()
        self.vgg = vgg_layers(style_layers + content_layers)
        self.style_layers = style_layers
        self.content_layers = content_layers
        self.vgg.trainable = False

    def call(self, inputs):
        inputs = inputs * 255.0
        preprocessed_input = tf.keras.applications.vgg19.preprocess_input(inputs)
        outputs = self.vgg(preprocessed_input)
        style_outputs = outputs[:len(self.style_layers)]
        content_outputs = outputs[len(self.style_layers):]

        style_outputs = [gram_matrix(style_output) for style_output in style_outputs]
        content_dict = {content_name:value for content_name, value in zip(self.content_layers, content_outputs)}
        style_dict = {style_name:value for style_name, value in zip(self.style_layers, style_outputs)}

        return {'content': content_dict, 'style': style_dict}
# ===============================
# 🏋️ TRAINING SETUP
# ===============================
extractor = StyleContentModel(style_layers, content_layers)
style_targets = extractor(style_image)['style']
content_targets = extractor(content_image)['content']

generated_image = tf.Variable(content_image)

optimizer = tf.optimizers.Adam(learning_rate=0.02)

style_weight = 1e-2
content_weight = 1e4

@tf.function()
def train_step(image):
    with tf.GradientTape() as tape:
        outputs = extractor(image)
        style_loss = tf.add_n([tf.reduce_mean((outputs['style'][name] - style_targets[name])**2)
                               for name in outputs['style']])
        style_loss *= style_weight / num_style_layers

        content_loss = tf.add_n([tf.reduce_mean((outputs['content'][name] - content_targets[name])**2)
                                 for name in outputs['content']])
        content_loss *= content_weight / num_content_layers

        loss = style_loss + content_loss

    grad = tape.gradient(loss, image)
    optimizer.apply_gradients([(grad, image)])
    image.assign(tf.clip_by_value(image, 0.0, 1.0))


In [ ]:
# ===============================
# 🚀 RUN STYLE TRANSFER
# ===============================
epochs = 10
steps_per_epoch = 100

start = time.time()
step = 0
for n in range(epochs):
    for m in range(steps_per_epoch):
        step += 1
        train_step(generated_image)
    print(f"Epoch {n+1} done")

end = time.time()
print(f"Total time: {end - start:.1f} seconds")
# ===============================
# 📸 DISPLAY RESULT
# ===============================
plt.figure(figsize=(10, 5))
plt.subplot(1, 3, 1)
imshow(content_image, "Content")
plt.subplot(1, 3, 2)
imshow(style_image, "Style")
plt.subplot(1, 3, 3)
imshow(generated_image, "Stylized")
plt.show()

# Save the final image
tensor_to_image(generated_image).save("stylized_output.png")
